# Bonus: Positional Embedding Ablation Evaluation Suite (Kaggle)
This notebook loads the saved  checkpoint from Kaggle inputs/working directory and runs the full Phase 2 evaluation suite in ~2 minutes.

**Evaluates:**
1. **Intrinsic Metrics:** Loss, Perplexity (PPL), Bits-Per-Byte (BPB)
2. **Generation Metrics:** BLEU-4, chrF, ROUGE-L (Greedy & Temperatures 0.5, 1.0, 1.5)
3. **Diversity Diagnostics:** Distinct-1, Distinct-2, 4-gram Repetition Rate
4. **Attention Diagnostics:** Layer 0, Layer 3, Layer 5 Heatmaps, Attention Entropy & Distance Plot
5. **Artifact Exports:** , , plots in 

In [40]:
!pip install -q sacrebleu rouge_score seaborn matplotlib huggingface_hub datasets


In [41]:
import os, sys, json, re, math, string, tarfile, glob, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from tqdm import tqdm
import sacrebleu
from rouge_score import rouge_scorer
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import hf_hub_download

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Running on device: {device}")


[Setup] Running on device: cuda


In [42]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_len, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.dk = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        
        mask = torch.triu(torch.ones(context_len, context_len) * float("-inf"), diagonal=1)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, d_model = x.size()
        
        q = self.W_q(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.dk)
        scores = scores + self.mask[:T, :T]
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        
        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).contiguous().view(B, T, d_model)
        
        out = self.resid_drop(self.W_o(context))
        return out, attn_weights

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = F.gelu(self.linear1(x))
        x = self.linear2(x)
        return self.dropout(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, context_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadCausalAttention(d_model, n_heads, context_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
    def forward(self, x):
        attn_out, attn_weights = self.attn(self.ln1(x))
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, attn_weights

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=6, d_ff=1024, context_len=256, dropout=0.1, pad_token_id=0):
        super().__init__()
        self.context_len = context_len
        self.pad_token_id = pad_token_id
        
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        # ABLATION: Removed self.pos_emb
        self.drop = nn.Dropout(dropout)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, context_len, dropout)
            for _ in range(n_layers)
        ])
        
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_emb.weight  # Weight tying
        
    def forward(self, idx, targets=None, return_attention=False):
        B, T = idx.size()
        assert T <= self.context_len, f"Sequence length {T} exceeds context length {self.context_len}"
        
        tok_emb = self.token_emb(idx)
        # ABLATION: Removed pos_emb
        x = self.drop(tok_emb)
        
        attention_maps = []
        for block in self.blocks:
            x, attn_weights = block(x)
            if return_attention:
                attention_maps.append(attn_weights)
                
        x = self.ln_f(x)
        logits = torch.nn.functional.linear(x, self.token_emb.weight)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=self.pad_token_id)
            
        if return_attention:
            return logits, loss, attention_maps
        return logits, loss

    @torch.no_grad()
    def greedy_generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.context_len else idx[:, -self.context_len:]
            logits, _ = self(idx_cond)
            idx_next = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.context_len else idx[:, -self.context_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [43]:
class TokenizerWrapper:
    def __init__(self, tok_json_path):
        with open(tok_json_path, encoding="utf-8") as f:
            data = json.load(f)
        self.vocab = data["vocab"]
        self.id_to_token = {v: k for k, v in self.vocab.items()}
        merges = [((a, b), a + b) for a, b in data["merges"]]
        self.merge_ranks = {p: i for i, (p, _) in enumerate(merges)}
        self.b2c = {i: chr(i) for i in range(256)}
        self.c2b = {chr(i): i for i in range(256)}
        punct = re.escape(string.punctuation + "।॥''""")
        self.word_pat = re.compile(rf" ?[^{punct}\s]+| ?[{punct}]+", re.UNICODE)

    def encode(self, text):
        out = []
        for w in self.word_pat.findall(text):
            s = [self.b2c[b] for b in w.encode("utf-8")]
            while len(s) >= 2:
                pairs = [(self.merge_ranks.get((s[i], s[i+1]), float("inf")), i)
                         for i in range(len(s)-1)]
                min_rank, i = min(pairs, key=lambda p: p[0])
                if min_rank == float("inf"): break
                s = s[:i] + [s[i] + s[i+1]] + s[i+2:]
            out.extend(self.vocab[t] for t in s if t in self.vocab)
        return out

    def decode(self, ids):
        toks = [self.id_to_token.get(i, "") for i in ids]
        raw = "".join(toks)
        b = bytes([self.c2b[c] for c in raw if c in self.c2b])
        return b.decode("utf-8", errors="replace")

class MemmapDataset(IterableDataset):
    def __init__(self, bin_path, context_len=256):
        self.data = np.memmap(bin_path, dtype=np.uint16, mode="r")
        self.context_len = context_len
    def __iter__(self):
        n = len(self.data)
        i = 0
        while i + self.context_len + 1 <= n:
            chunk = self.data[i:i+self.context_len+1]
            x = torch.from_numpy(chunk[:-1].astype(np.int64))
            y = torch.from_numpy(chunk[1:].astype(np.int64))
            yield x, y
            i += self.context_len + 1


In [44]:
def evaluate_ppl_bpb(model, val_loader, device, tokenizer, num_batches=200):
    model.eval()
    total_loss_nats = 0.0
    total_tokens = 0
    total_bytes = 0
    val_iter = iter(val_loader)
    with torch.no_grad():
        for i in tqdm(range(num_batches), desc="Evaluating PPL/BPB"):
            try:
                x, y = next(val_iter)
            except StopIteration:
                break
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, targets=y)
            n_toks = y.numel()
            total_loss_nats += loss.item() * n_toks
            total_tokens += n_toks
            flat_ids = y.view(-1).cpu().tolist()
            decoded_text = tokenizer.decode(flat_ids)
            total_bytes += len(decoded_text.encode("utf-8"))
    avg_loss = total_loss_nats / total_tokens if total_tokens > 0 else 0.0
    ppl = math.exp(min(avg_loss, 20))
    avg_bytes_per_token = total_bytes / total_tokens if total_tokens > 0 else 1.0
    bpb = (avg_loss / math.log(2)) / avg_bytes_per_token
    return avg_loss, ppl, bpb, avg_bytes_per_token

def compute_repetition_rate(text, n=4):
    tokens = text.split()
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    repeated = len(ngrams) - len(set(ngrams))
    return repeated / len(ngrams) if ngrams else 0.0

def evaluate_generation(model, tokenizer, device, test_prefixes, references):
    model.eval()
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    results = {}

    greedy_generations = []
    for prefix in tqdm(test_prefixes, desc="Greedy Generation"):
        enc = tokenizer.encode(prefix)
        if not enc: enc = [1]
        x = torch.tensor([enc], dtype=torch.long).to(device)
        with torch.no_grad():
            gen_idx = model.greedy_generate(x, max_new_tokens=50)
        greedy_generations.append(tokenizer.decode(gen_idx[0].cpu().tolist()))

    bleu_g = sacrebleu.corpus_bleu(greedy_generations, [references]).score
    chrf_g = sacrebleu.corpus_chrf(greedy_generations, [references]).score
    rougeL_g = sum(rouge.score(r, g)["rougeL"].fmeasure for r, g in zip(references, greedy_generations)) / max(len(greedy_generations), 1)
    all_w_g = [w for g in greedy_generations for w in g.split()]
    dist1_g = len(set(all_w_g)) / len(all_w_g) if all_w_g else 0.0
    bg = list(zip(all_w_g, all_w_g[1:]))
    dist2_g = len(set(bg)) / len(bg) if bg else 0.0
    rep_g = sum(compute_repetition_rate(g, 4) for g in greedy_generations) / max(len(greedy_generations), 1)

    results["greedy"] = {
        "BLEU-4": round(bleu_g, 4), "chrF": round(chrf_g, 4), "ROUGE-L": round(rougeL_g, 4),
        "Distinct-1": round(dist1_g, 4), "Distinct-2": round(dist2_g, 4),
        "repetition_rate_4gram": round(rep_g, 4), "generations": greedy_generations,
    }

    for temp in [0.5, 1.0, 1.5]:
        generations = []
        for prefix in tqdm(test_prefixes, desc=f"Sampling Temp {temp}"):
            enc = tokenizer.encode(prefix)
            if not enc: enc = [1]
            x = torch.tensor([enc], dtype=torch.long).to(device)
            with torch.no_grad():
                gen_idx = model.generate(x, max_new_tokens=50, temperature=temp)
            generations.append(tokenizer.decode(gen_idx[0].cpu().tolist()))

        bleu = sacrebleu.corpus_bleu(generations, [references]).score
        chrf = sacrebleu.corpus_chrf(generations, [references]).score
        rougeL = sum(rouge.score(r, g)["rougeL"].fmeasure for r, g in zip(references, generations)) / max(len(generations), 1)
        all_words = [w for gen in generations for w in gen.split()]
        dist1 = len(set(all_words)) / len(all_words) if all_words else 0.0
        bigrams_all = list(zip(all_words, all_words[1:]))
        dist2 = len(set(bigrams_all)) / len(bigrams_all) if bigrams_all else 0.0
        rep_rate = sum(compute_repetition_rate(gen, 4) for gen in generations) / max(len(generations), 1)

        results[float(temp)] = {
            "BLEU-4": round(bleu, 4), "chrF": round(chrf, 4), "ROUGE-L": round(rougeL, 4),
            "Distinct-1": round(dist1, 4), "Distinct-2": round(dist2, 4),
            "repetition_rate_4gram": round(rep_rate, 4), "generations": generations,
        }
    return results

def compute_attention_stats(attention_maps):
    entropy_table, distance_table = [], []
    for layer_idx, attn in enumerate(attention_maps):
        attn_np = attn[0].detach().cpu().numpy()
        n_heads, T, _ = attn_np.shape
        ent_row, dist_row = [], []
        for h in range(n_heads):
            weights = attn_np[h]
            eps = 1e-9
            ent = -np.sum(weights * np.log(weights + eps), axis=-1).mean()
            ent_row.append(float(ent))
            positions = np.arange(T)
            dists = [np.sum(np.abs(positions - q) * weights[q]) for q in range(T)]
            dist_row.append(float(np.mean(dists)))
        entropy_table.append(ent_row)
        distance_table.append(dist_row)
    return entropy_table, distance_table

def plot_attention(model, tokenizer, device, text, save_dir):
    model.eval()
    enc = tokenizer.encode(text)
    if not enc: return {}
    x = torch.tensor([enc], dtype=torch.long).to(device)
    with torch.no_grad():
        _, _, attention_maps = model(x, return_attention=True)
    tokens = [tokenizer.decode([t]) for t in x[0].tolist()]
    n_layers = len(attention_maps)
    n_heads = attention_maps[0].shape[1]

    layer_indices = sorted(set([0, n_layers // 2, n_layers - 1]))
    for layer_idx in layer_indices:
        attn = attention_maps[layer_idx][0].detach().cpu().numpy()
        cols = min(n_heads, 4)
        fig, axes = plt.subplots(1, cols, figsize=(5 * cols, 5))
        if cols == 1: axes = [axes]
        for h in range(cols):
            ax = axes[h]
            sns.heatmap(attn[h], ax=ax, cmap="viridis", xticklabels=tokens, yticklabels=tokens, vmin=0, vmax=attn[h].max())
            ax.set_title(f"Layer {layer_idx} Head {h}")
            ax.set_xlabel("Key position"); ax.set_ylabel("Query position")
            plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
            plt.setp(ax.get_yticklabels(), rotation=0, fontsize=7)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"attn_layer{layer_idx}.png"), dpi=150)
        plt.close()

    entropy_table, distance_table = compute_attention_stats(attention_maps)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.heatmap(np.array(entropy_table), ax=axes[0], cmap="magma", xticklabels=[f"H{h}" for h in range(n_heads)], yticklabels=[f"L{l}" for l in range(n_layers)], annot=True, fmt=".2f")
    axes[0].set_title("Attention Entropy (nats)")
    axes[0].set_xlabel("Head"); axes[0].set_ylabel("Layer")
    sns.heatmap(np.array(distance_table), ax=axes[1], cmap="viridis", xticklabels=[f"H{h}" for h in range(n_heads)], yticklabels=[f"L{l}" for l in range(n_layers)], annot=True, fmt=".1f")
    axes[1].set_title("Mean Attention Distance (tokens)")
    axes[1].set_xlabel("Head"); axes[1].set_ylabel("Layer")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "attn_entropy_distance.png"), dpi=150)
    plt.close()
    return {"entropy_per_layer_head": entropy_table, "distance_per_layer_head": distance_table}


In [45]:
OUT_DIR = "bonus_ablation_output"
os.makedirs(OUT_DIR, exist_ok=True)

candidates = []
for root in ["/kaggle/input", "/kaggle/working", "."]:
    if os.path.exists(root):
        for path in glob.glob(os.path.join(root, "**", "*model_ablated*"), recursive=True):
            candidates.append(path)

print(f"[Search] Found candidate paths: {candidates}")

ckpt_path = None
for c in candidates:
    if os.path.isfile(c) or (os.path.isdir(c) and (os.path.exists(os.path.join(c, "data.pkl")) or os.path.exists(os.path.join(c, "model_ablated.pt")))):
        ckpt_path = c
        break

if not ckpt_path:
    dataset_dir = "/kaggle/input/datasets/manyajain3133/bonus-evaluate/model_ablated"
    if os.path.exists(dataset_dir):
        ckpt_path = dataset_dir
    else:
        fallback = glob.glob("/kaggle/input/**/bonus-evaluate/**/model_ablated*", recursive=True)
        if fallback:
            ckpt_path = fallback[0]

print(f"[Checkpoint] Selected model path: {ckpt_path}")

# Fix Kaggle unzipping issue with PyTorch's required 'archive/' prefix:
if os.path.isdir(ckpt_path) and os.path.exists(os.path.join(ckpt_path, "data.pkl")):
    print("[Checkpoint] Path is a Kaggle-extracted directory. Repacking into a .pt zip file...")
    rezipped_path = "/kaggle/working/model_ablated_rezipped.pt"
    with zipfile.ZipFile(rezipped_path, "w", zipfile.ZIP_STORED) as zipf:
        for root, dirs, files in os.walk(ckpt_path):
            for file in files:
                file_path = os.path.join(root, file)
                rel_path = os.path.relpath(file_path, ckpt_path)
                arcname = os.path.join("archive", rel_path)
                zipf.write(file_path, arcname)
    print(f"[Checkpoint] Repacked successfully to {rezipped_path}")
    ckpt_path = rezipped_path

model = GPTLanguageModel(vocab_size=16240, d_model=256, n_heads=8, n_layers=6, d_ff=1024, context_len=256)
try:
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print("[Checkpoint] Loaded successfully with torch.load!")
except Exception as e:
    print(f"[Checkpoint] Direct load warning: {e}. Trying weights file inside folder...")
    inner_file = os.path.join(ckpt_path, "model_ablated.pt")
    if os.path.exists(inner_file):
        model.load_state_dict(torch.load(inner_file, map_location=device))
        print("[Checkpoint] Inner model_ablated.pt loaded successfully!")
    else:
        raise e

model.to(device)
model.eval()

[Search] Found candidate paths: ['/kaggle/input/datasets/manyajain3133/bonus-evaluate/model_ablated', '/kaggle/working/model_ablated.pt', '/kaggle/working/model_ablated_rezipped.pt', './model_ablated.pt', './model_ablated_rezipped.pt']
[Checkpoint] Selected model path: /kaggle/input/datasets/manyajain3133/bonus-evaluate/model_ablated
[Checkpoint] Path is a Kaggle-extracted directory. Repacking into a .pt zip file...
[Checkpoint] Repacked successfully to /kaggle/working/model_ablated_rezipped.pt
[Checkpoint] Loaded successfully with torch.load!


GPTLanguageModel(
  (token_emb): Embedding(16240, 256, padding_idx=0)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-5): 6 x TransformerBlock(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadCausalAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_features=256, bias=True)
        (W_v): Linear(in_features=256, out_features=256, bias=True)
        (W_o): Linear(in_features=256, out_features=256, bias=True)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((256,), eps=1e-05

In [50]:
TOK_PATH = None
for p in ["repo/phase2/tokenizers/hindi_tokenizer_16000.json", "phase2/tokenizers/hindi_tokenizer_16000.json", "checkpoints/pretrained/hindi/tokenizer.json"]:
    if os.path.exists(p):
        TOK_PATH = p
        break
if not TOK_PATH:
    tok_search = glob.glob("/kaggle/input/**/hindi_tokenizer*.json", recursive=True) + glob.glob("/kaggle/working/**/tokenizer*.json", recursive=True)
    if tok_search:
        TOK_PATH = tok_search[0]
    else:
        print("[Tokenizer] Downloading hindi tokenizer from HF Hub...")
        TOK_PATH = hf_hub_download(repo_id="manyajain111/lma-phase2", filename="hindi/tokenizer.json", local_dir="/kaggle/working/checkpoints/pretrained")

val_bin = None
for p in ["phase2/data/hindi/hindi_val.bin", "hindi_val.bin", "/kaggle/working/hindi_val.bin"]:
    if os.path.exists(p):
        val_bin = p
        break
if not val_bin:
    bin_search = glob.glob("/kaggle/input/**/hindi_val.bin", recursive=True) + glob.glob("/kaggle/working/**/hindi_val.bin", recursive=True)
    if bin_search:
        val_bin = bin_search[0]
    else:
        print("[Dataset] Downloading dataset from HF Hub...")
        tar_path = hf_hub_download(repo_id="manyajain111/hindi-lma-corpus", filename="hindi_submission_phase1.tar.gz", repo_type="dataset")
        print("[Dataset] Extracting tar archive...")
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path="/kaggle/working/")
        extracted = glob.glob("/kaggle/working/**/hindi_val.bin", recursive=True)
        val_bin = extracted[0] if extracted else "/kaggle/working/hindi_val.bin"

print(f"[Tokenizer Path]: {TOK_PATH}")
print(f"[Val Dataset Path]: {val_bin}")
tokenizer = TokenizerWrapper(TOK_PATH)

[Dataset] Downloading dataset from HF Hub...
[Dataset] Extracting tar archive...


/tmp/ipykernel_58/2238072711.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="/kaggle/working/")


[Tokenizer Path]: checkpoints/pretrained/hindi/tokenizer.json
[Val Dataset Path]: /kaggle/working/hindi_val.bin


In [53]:
import os, glob, numpy as np
from tqdm import tqdm

# 1. Find the extracted text files
val_txt_paths = glob.glob("/kaggle/working/**/val.txt", recursive=True)
test_txt_paths = glob.glob("/kaggle/working/**/test.txt", recursive=True)

def create_bin_from_text(txt_path, bin_path, tokenizer):
    print(f"Tokenizing {txt_path}...")
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()
    
    # Add caching for massive speedup (100x faster)
    cache = {}
    tokens = []
    
    words = tokenizer.word_pat.findall(text)
    
    for w in tqdm(words, desc=f"Encoding {os.path.basename(txt_path)}"):
        if w not in cache:
            # Only calculate merges for unseen words
            s = [tokenizer.b2c[b] for b in w.encode("utf-8")]
            while len(s) >= 2:
                pairs = [(tokenizer.merge_ranks.get((s[i], s[i+1]), float("inf")), i)
                         for i in range(len(s)-1)]
                min_rank, i = min(pairs, key=lambda p: p[0])
                if min_rank == float("inf"): break
                s = s[:i] + [s[i] + s[i+1]] + s[i+2:]
            cache[w] = [tokenizer.vocab[t] for t in s if t in tokenizer.vocab]
            
        tokens.extend(cache[w])
    
    # Save as raw uint16 binary (no numpy headers)
    arr = np.array(tokens, dtype=np.uint16)
    arr.tofile(bin_path)
    print(f"✅ Saved {len(tokens):,} tokens to {bin_path}")

# 2. Tokenize and save Validation
if val_txt_paths:
    create_bin_from_text(val_txt_paths[0], "/kaggle/working/hindi_val.bin", tokenizer)
else:
    print("❌ Could not find val.txt in /kaggle/working/!")

# 3. Tokenize and save Test
if test_txt_paths:
    create_bin_from_text(test_txt_paths[0], "/kaggle/working/hindi_test.bin", tokenizer)
else:
    print("❌ Could not find test.txt in /kaggle/working/!")

Tokenizing /kaggle/working/datasets/val.txt...


Encoding val.txt: 100%|██████████| 25008266/25008266 [01:29<00:00, 277897.10it/s]


✅ Saved 29,106,897 tokens to /kaggle/working/hindi_val.bin
Tokenizing /kaggle/working/datasets/test.txt...


Encoding test.txt: 100%|██████████| 24878187/24878187 [01:28<00:00, 280361.87it/s]


✅ Saved 28,906,057 tokens to /kaggle/working/hindi_test.bin


In [54]:
print("=========================================")
print(" RUNNING ABLATION EVALUATION SUITE ")
print("=========================================")

val_ds = MemmapDataset(val_bin, context_len=256)
val_loader = DataLoader(val_ds, batch_size=32)
val_loss, ppl, bpb, avg_bytes = evaluate_ppl_bpb(model, val_loader, device, tokenizer, num_batches=200)

print(f"\n[INTRINSIC METRICS]")
print(f"  Val Loss : {val_loss:.4f} nats")
print(f"  Perplexity (PPL) : {ppl:.2f}")
print(f"  Bits-Per-Byte (BPB) : {bpb:.4f}")
print(f"  Avg Bytes/Token : {avg_bytes:.2f}")

prefixes = ["मैं आज", "भारत के", "वह एक", "सरकार ने", "देश में"]
references = ["मैं आज बाजार गया था।", "भारत के प्रधानमंत्री ने कहा", "वह एक बहुत ही अच्छा इंसान है।", "सरकार ने नई नीति की घोषणा की।", "देश में लोकतंत्र की स्थापना हुई।"]
gen_results = evaluate_generation(model, tokenizer, device, prefixes, references)

print(f"\n[GENERATION METRICS]")
for mode, res in gen_results.items():
    print(f"  Mode/Temp {mode}: BLEU-4={res['BLEU-4']:.2f} | chrF={res['chrF']:.2f} | ROUGE-L={res['ROUGE-L']:.4f} | Dist-1={res['Distinct-1']:.4f} | Dist-2={res['Distinct-2']:.4f} | 4gram-Rep={res['repetition_rate_4gram']:.4f}")

attn_text = "मैं आज बाजार गया था।"
attn_stats = plot_attention(model, tokenizer, device, attn_text, OUT_DIR)
print(f"\n[ATTENTION DIAGNOSTICS] Heatmaps and entropy/distance plots saved to {OUT_DIR}/")

serialisable_gen = {}
for temp, res in gen_results.items():
    r = dict(res)
    r.pop("generations", None)
    serialisable_gen[str(temp)] = r

with open(os.path.join(OUT_DIR, "generated_samples.json"), "w", encoding="utf-8") as f:
    json.dump({str(t): res["generations"] for t, res in gen_results.items()}, f, ensure_ascii=False, indent=2)

results = {
    "language": "hindi",
    "ablation": "no_positional_embeddings",
    "checkpoint": ckpt_path,
    "val_loss": round(val_loss, 4),
    "ppl": round(ppl, 2),
    "bpb": round(bpb, 4),
    "avg_bytes_per_token": round(avg_bytes, 4),
    "generation_metrics": serialisable_gen,
    "attention_stats": attn_stats,
}
with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"\n SUCCESS! All evaluation metrics and plots saved to {OUT_DIR}/")


 RUNNING ABLATION EVALUATION SUITE 


Evaluating PPL/BPB: 100%|██████████| 200/200 [00:15<00:00, 12.83it/s]



[INTRINSIC METRICS]
  Val Loss : 4.0697 nats
  Perplexity (PPL) : 58.54
  Bits-Per-Byte (BPB) : 0.5674
  Avg Bytes/Token : 10.35


Sampling Temp 1.5: 100%|██████████| 5/5 [00:00<00:00,  5.02it/s]
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Glyph 2350 (\N{DEVANAGARI LETTER MA}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Matplotlib currently does not support Devanagari natively.
  fig.canvas.draw()
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Glyph 2376 (\N{DEVANAGARI VOWEL SIGN AI}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Glyph 2306 (\N{DEVANAGARI SIGN ANUSVARA}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Glyph 2310 (\N{DEVANAGARI LETTER AA}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Glyph 2332 (\N{DEVANAGARI LETTER JA}


[GENERATION METRICS]
  Mode/Temp greedy: BLEU-4=1.08 | chrF=11.28 | ROUGE-L=0.0000 | Dist-1=0.1703 | Dist-2=0.2412 | 4gram-Rep=0.6852
  Mode/Temp 0.5: BLEU-4=0.73 | chrF=13.07 | ROUGE-L=0.0000 | Dist-1=0.4911 | Dist-2=0.8296 | 4gram-Rep=0.0526
  Mode/Temp 1.0: BLEU-4=0.86 | chrF=13.11 | ROUGE-L=0.0000 | Dist-1=0.7677 | Dist-2=0.9746 | 4gram-Rep=0.0043
  Mode/Temp 1.5: BLEU-4=0.87 | chrF=12.59 | ROUGE-L=0.0000 | Dist-1=0.9185 | Dist-2=1.0000 | 4gram-Rep=0.0000


/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Glyph 2350 (\N{DEVANAGARI LETTER MA}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Matplotlib currently does not support Devanagari natively.
  fig.canvas.draw()
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Glyph 2376 (\N{DEVANAGARI VOWEL SIGN AI}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Glyph 2306 (\N{DEVANAGARI SIGN ANUSVARA}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Glyph 2310 (\N{DEVANAGARI LETTER AA}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/usr/local/lib/python3.12/dist-packages/seaborn/utils.py:61: UserWarning: Glyph 2332 (\N{DEVANAGARI LETTER JA}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
/usr/loca


[ATTENTION DIAGNOSTICS] Heatmaps and entropy/distance plots saved to bonus_ablation_output/

 SUCCESS! All evaluation metrics and plots saved to bonus_ablation_output/
